# Trapped-Ion QCCD Framework — End-to-End Demo

This notebook walks through the full trapped-ion QCCD simulation pipeline:

1. **Build** an ideal QEC circuit (d=2 rotated surface code)
2. **Compile** it to native trapped-ion operations on two architectures:
   - **Augmented Grid** — fast greedy routing
   - **WISE** — SAT-based optimal routing (slower, higher quality)
3. **Visualise** the architecture layout (static figures)
4. **Animate** ion transport during gate execution (inline video)
5. **Apply hardware noise** (MS gate, shuttling, measurement errors)
6. **Decode** with PyMatching and measure logical error rates

## 0 — Imports & Setup

In [ ]:
from __future__ import annotations
import time, warnings
import numpy as np
import matplotlib.pyplot as plt
import stim

# QECToStim  ──────────────────────────────────────────────────────────
from qectostim.codes.surface.rotated_surface import RotatedSurfaceCode
from qectostim.experiments.memory import CSSMemoryExperiment

# Trapped-ion module  ─────────────────────────────────────────────────
from qectostim.experiments.hardware_simulation.trapped_ion.utils import (
    AugmentedGridArchitecture,
    WISEArchitecture,
    TrappedIonCompiler,
    TrappedIonExperiment,
    TrappedIonNoiseModel,
)
from qectostim.experiments.hardware_simulation.trapped_ion.utils.qccd_nodes import (
    QCCDWiseArch,
    SpectatorIon,
)
from qectostim.experiments.hardware_simulation.trapped_ion.viz import (
    display_architecture,
    animate_transport,
    render_animation,
)
from qectostim.experiments.hardware_simulation.trapped_ion.demo.run import (
    build_viz_mappings,
)

# inline plotting
%matplotlib inline
plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore", category=DeprecationWarning)
print("✓ imports OK")

## 1 — Build an Ideal QEC Circuit

We use a **d=2 rotated surface code** with 1 round of syndrome extraction in the Z basis.
This produces a small stim circuit that we'll then compile to trapped-ion hardware.

In [ ]:
code = RotatedSurfaceCode(distance=2)
mem = CSSMemoryExperiment(code=code, rounds=1, noise_model=None, basis="z")
ideal = mem.to_stim()

print(f"Code:      d={code.distance}, n={code.n}")
print(f"Circuit:   {len(ideal)} instructions, {ideal.num_qubits} qubits")
print()
print(ideal)

## 2 — Augmented Grid Architecture

The **Augmented Grid** architecture uses a regular grid of ion traps connected by
junctions. Routing is done with a fast greedy algorithm. Compilation typically
takes < 1 second.

### 2.1 — Compile

In [ ]:
# Build architecture
ag_arch = AugmentedGridArchitecture(trap_capacity=2, rows=3, cols=3, padding=0)
ag_compiler = TrappedIonCompiler(ag_arch, is_wise=False, show_progress=True)

# Compile
t0 = time.perf_counter()
ag_compiled = ag_compiler.compile(ideal)
t1 = time.perf_counter()

# Extract viz mappings
ag_roles, ag_p2l, ag_remap = build_viz_mappings(ag_compiler)

# Metrics
ag_batches = ag_compiled.scheduled.batches
ag_metrics = ag_compiled.compute_metrics()

print(f"Compile time:     {t1 - t0:.2f} s")
print(f"Traps:            {len(ag_arch._manipulationTraps)}")
print(f"Parallel batches: {len(ag_batches)}")
print(f"Total operations: {ag_metrics.get('total_operations', '?')}")
print(f"Depth:            {ag_metrics.get('depth', '?')}")
print(f"Duration (µs):    {ag_metrics.get('duration_us', '?')}")
print(f"Ion roles:        {ag_roles}")

### 2.2 — Static Layout

Display the trap grid with ions colour-coded by role:
- **D** (blue) = data qubit
- **M** (red) = measurement/ancilla qubit  
- **P** (grey) = spectator/parking ion

In [ ]:
ag_arch.resetArrangement()
ag_arch.refreshGraph()

fig, ax = display_architecture(
    ag_arch,
    title="Augmented Grid — initial layout",
    show_junctions=True,
    show_edges=True,
    show_ions=True,
    show_labels=False,
    show_legend=True,
    ion_roles=ag_roles,
    ion_idx_remap=ag_remap,
    physical_to_logical=ag_p2l,
)
plt.show()

### 2.3 — Ion Transport Animation

Animate the first 10 parallel batches of ion shuttling and gate execution.
Each frame shows ions moving between traps as the compiled circuit executes.

In [ ]:
ag_arch.resetArrangement()
ag_arch.refreshGraph()

ag_anim = animate_transport(
    arch=ag_arch,
    operations=ag_batches[:10],   # first 10 batches for speed
    interval=600,
    show_labels=False,
    interp_frames=2,
    gate_hold_frames=1,
    stim_circuit=ideal,
    figsize=(14, 9),
    title_prefix="Augmented Grid",
    repeat=False,
    ion_roles=ag_roles,
    ion_idx_remap=ag_remap,
    physical_to_logical=ag_p2l,
)

# Display inline (HTML5 video)
from IPython.display import HTML
HTML(ag_anim.to_jshtml())

## 3 — WISE Architecture

The **WISE** (Window-based Ion Shuttling Engine) architecture uses a SAT solver
to compute optimal ion routing through an odd-even-sorter network. This produces
higher-quality schedules but takes ~45 seconds for d=2.

A tqdm progress bar tracks the SAT solving rounds.

### 3.1 — Compile

In [ ]:
# Build WISE architecture  (m×n grid with k ions per trap)
wise_cfg = QCCDWiseArch(m=5, n=5, k=2)
wise_arch = WISEArchitecture(
    wise_config=wise_cfg,
    add_spectators=True,
    compact_clustering=True,
)
wise_compiler = TrappedIonCompiler(
    wise_arch, is_wise=True, wise_config=wise_cfg, show_progress=True,
)

# Compile  (expect ~45 s for d=2; progress bar tracks SAT rounds)
t0 = time.perf_counter()
wise_compiled = wise_compiler.compile(ideal)
t1 = time.perf_counter()

# Extract viz mappings
wise_roles, wise_p2l, wise_remap = build_viz_mappings(wise_compiler)

# Metrics
wise_batches = wise_compiled.scheduled.batches
wise_metrics = wise_compiled.compute_metrics()

print(f"\nCompile time:     {t1 - t0:.2f} s")
print(f"Traps:            {len(wise_arch._manipulationTraps)}")
print(f"Parallel batches: {len(wise_batches)}")
print(f"Total operations: {wise_metrics.get('total_operations', '?')}")
print(f"Depth:            {wise_metrics.get('depth', '?')}")
print(f"Duration (µs):    {wise_metrics.get('duration_us', '?')}")
print(f"Ion roles:        {wise_roles}")

### 3.2 — Static Layout

In [ ]:
wise_arch.resetArrangement()
wise_arch.refreshGraph()

fig, ax = display_architecture(
    wise_arch,
    title="WISE — initial layout",
    show_junctions=True,
    show_edges=True,
    show_ions=True,
    show_labels=True,
    show_legend=True,
    ion_roles=wise_roles,
    ion_idx_remap=wise_remap,
    physical_to_logical=wise_p2l,
)
plt.show()

### 3.3 — Ion Transport Animation

In [ ]:
wise_arch.resetArrangement()
wise_arch.refreshGraph()

wise_anim = animate_transport(
    arch=wise_arch,
    operations=wise_batches[:10],
    interval=600,
    show_labels=True,
    interp_frames=2,
    gate_hold_frames=1,
    stim_circuit=ideal,
    figsize=(16, 12),
    title_prefix="WISE",
    repeat=False,
    ion_roles=wise_roles,
    ion_idx_remap=wise_remap,
    physical_to_logical=wise_p2l,
)

from IPython.display import HTML
HTML(wise_anim.to_jshtml())

### 3.4 — Save Animation to MP4

Requires `ffmpeg` to be installed (`brew install ffmpeg` on macOS).
Set `SAVE_MP4 = True` to write the file.

In [ ]:
SAVE_MP4 = False   # ← set True to write MP4 files

if SAVE_MP4:
    # Re-create animations with all batches (not just first 10)
    ag_arch.resetArrangement(); ag_arch.refreshGraph()
    ag_full_anim = animate_transport(
        arch=ag_arch, operations=ag_batches, interval=600,
        show_labels=False, interp_frames=2, gate_hold_frames=1,
        stim_circuit=ideal, figsize=(16, 10),
        title_prefix="Augmented Grid", repeat=False,
        ion_roles=ag_roles, ion_idx_remap=ag_remap,
        physical_to_logical=ag_p2l,
    )
    render_animation(ag_full_anim, "aug_grid_transport.mp4", fps=5, dpi=120)
    print("✓ Saved aug_grid_transport.mp4")

    wise_arch.resetArrangement(); wise_arch.refreshGraph()
    wise_full_anim = animate_transport(
        arch=wise_arch, operations=wise_batches, interval=600,
        show_labels=True, interp_frames=2, gate_hold_frames=1,
        stim_circuit=ideal, figsize=(16, 12),
        title_prefix="WISE", repeat=False,
        ion_roles=wise_roles, ion_idx_remap=wise_remap,
        physical_to_logical=wise_p2l,
    )
    render_animation(wise_full_anim, "wise_transport.mp4", fps=5, dpi=120)
    print("✓ Saved wise_transport.mp4")
else:
    print("Skipping MP4 export (set SAVE_MP4 = True to enable)")

## 4 — Hardware Noise & Decoding

Apply a **trapped-ion noise model** (MS gate errors, shuttling decoherence,
measurement infidelity) to the compiled circuit, then decode with
**PyMatching** to measure the logical error rate.

We use the Augmented Grid compiled circuit for speed.

In [ ]:
# Fresh architecture to avoid stale ion references
noise_arch = AugmentedGridArchitecture(trap_capacity=2, rows=3, cols=3, padding=0)
noise_compiler = TrappedIonCompiler(noise_arch, is_wise=False, show_progress=False)
noise_model = TrappedIonNoiseModel()

experiment = TrappedIonExperiment(
    code=code,
    architecture=noise_arch,
    compiler=noise_compiler,
    hardware_noise=noise_model,
    rounds=1,
    basis="z",
)

# Build → compile → inject noise
ideal_circ = experiment.build_ideal_circuit()
experiment._compiled = experiment.compiler.compile(ideal_circ)
noisy_circuit = experiment.apply_hardware_noise()

print(f"Noisy circuit: {len(noisy_circuit)} instructions")
has_noise = any(kw in str(noisy_circuit) for kw in ("X_ERROR", "Z_ERROR", "DEPOLARIZE"))
print(f"Contains noise: {has_noise}")

### 4.1 — Decode with PyMatching

In [ ]:
from qectostim.decoders.pymatching_decoder import PyMatchingDecoder

NUM_SHOTS = 1_000

# Build detector error model
dem = noisy_circuit.detector_error_model(
    decompose_errors=True,
    approximate_disjoint_errors=True,
)
decoder = PyMatchingDecoder(dem=dem)

# Sample and decode
sampler = noisy_circuit.compile_detector_sampler()
detection_events, observable_flips = sampler.sample(
    NUM_SHOTS, separate_observables=True,
)
predictions = decoder.decode_batch(detection_events)
num_errors = int(np.sum(np.any(predictions != observable_flips, axis=1)))
ler = num_errors / NUM_SHOTS

print(f"Shots:              {NUM_SHOTS}")
print(f"Logical errors:     {num_errors}")
print(f"Logical error rate: {ler:.4f}")

## 5 — Architecture Comparison

Compare the compilation metrics of both architectures side by side.

In [ ]:
import pandas as pd

comparison = {
    "Metric": [
        "Traps",
        "Total operations",
        "Parallel batches",
        "Circuit depth",
        "Duration (µs)",
    ],
    "Augmented Grid": [
        len(ag_arch._manipulationTraps),
        ag_metrics.get("total_operations", "?"),
        len(ag_batches),
        ag_metrics.get("depth", "?"),
        ag_metrics.get("duration_us", "?"),
    ],
    "WISE": [
        len(wise_arch._manipulationTraps),
        wise_metrics.get("total_operations", "?"),
        len(wise_batches),
        wise_metrics.get("depth", "?"),
        wise_metrics.get("duration_us", "?"),
    ],
}

df = pd.DataFrame(comparison).set_index("Metric")
df

## 6 — Next Steps

- **Increase distance**: Try `d=3` or `d=4` (WISE compile time scales significantly)
- **Sweep noise**: Vary `TrappedIonNoiseModel` parameters to study thresholds
- **Custom circuits**: Pass any `stim.Circuit` to `compiler.compile()`
- **CLI runner**: `python -m qectostim.experiments.hardware_simulation.trapped_ion.demo.run --help`
- **Tests**: `pytest src/qectostim/experiments/hardware_simulation/trapped_ion/demo/test_e2e.py -v`